# Objective
This notebook compares the [BOOST](https://arxiv.org/pdf/2404.08826) scheduling policy with [PCS](https://www.usenix.org/conference/osdi24/presentation/bin-faisal).
The objective is to understand whether BOOST can provide similar predictability in job completion times as PCS.

Based on the preliminary experiment (described below), BOOST seems promising but PCS is able to achieve better results.

# Background

#### Predictability vs. Performance
In order for a scheduler to provide accurate job completion time predictions, we need to limit the degree of preemption experienced by jobs. By controlling the degree of preemption &mdash; from no preemption as in FIFO to *unbounded* preemption as in Shortest-Job-First (SJF) &mdash; we can achieve different trade-offs between predictability (low prediction error) and performance (low Job Completion Times (JCTs)).

The main question is: which scheduling policy offers the necessary flexibility?

#### PCS
The PCS paper argues to use Weighted-Fair-Queues (WFQs).
The main insight is that different WFQ configurations span the space of scheduling options and hence different objectives: from well-known options such as FIFO (WFQ with a single queue) and SJF (queues with small jobs assigned exponentially larger weights) to intermediate options.
PCS searches different WFQ configurations (e.g., number of queues, queue weights etc.) to achieve varying trade-offs between competing objectives.

#### BOOST
BOOST is designed to provide tail-optimal JCTs for light-tailed workloads (low variation in job-sizes) while improving upon FIFO (which has traditionally been known to be tail-optimal).
BOOST works similar to FIFO but in order to achieve its objective, it pretendes small jobs arrive earlier than their true arrival times; referred to as the *boosted arrival time*.
Thus jobs with earlier (smaller) boosted arrival times are served first.
The boosted arrival time of a job is:
$$
\text{boosted arrival time} = \text{arrival time} - b(s)
$$
Where $s$ is a job's size and $b(s)$ is the boost function.
The specific boost function the paper uses is:
$$
b(s) = \frac{1}{\gamma} \log \left( \frac{1}{1 - \exp(-\gamma \cdot s)} \right)
$$
Intuitively, smaller the job, larger the value of the boost function and hence smaller the boosted arrival time.
The scheme is parameterized by $\gamma$, which is a hyperparameter allowing BOOST to approximate extreme points like FIFO (larger $\gamma$) and SJF (smaller $\gamma$) as well as intermediate points.

# Experiment: PCS vs. BOOST - avg JCT vs. avg Prediction Error
The experiment uses simulations to evaluate the ability of both scheduling policies to achieve Pareto-optimal trade-offs between minimizing average JCTs and average prediction error.
The goal of the experiment is to see whether Pareto-optimal BOOST schedulers are *better* than Pareto-optimal WFQ configurations as in PCS.

For a given workload, PCS's simulation-based search strategy can identify Pareto-optimal WFQ configurations and Pareto-optimal $\gamma$ values for the BOOST policy.

#### Workloads and Setup
For the evaluation, two workloads A and B from the PCS paper (Workload 1 and 3, Table 1) are considered.
Workload A is light-tailed (squared coefficient of variation of 0.7) while workload B is heavy-tailed (squared coefficient of variation of 2.4).
For simplicity, the simulations are for 1-GPU running with the system load set to be 80%.

#### Metrics
The evaluation considers average JCT and average prediction error.
Prediction error is defined as:
$$
100.0 * (\text{True JCT} - \text{Predicted JCT}) / \text{Predicted JCT}
$$

#### Results
The plots below show the trade-offs achieved by different WFQ and BOOST configurations as well as FIFO (0% avg prediction error) and SJF (lowest avg JCT) for both workloads A and B.

<!-- <div style="display: flex; align-items: flex-start;">
    <div style="text-align: center; margin-right: 10px;">
        <p>Workload A (light-tailed)</p>
        <img src="evaluated_pareto_front_avg_jct_avg_pred_error_themis1.png" alt="Workload1" style="width: 500px;">
    </div>
    <div style="text-align: center;">
        <p>Workload B (heavy-tailed)</p>
        <img src="evaluated_pareto_front_avg_jct_avg_pred_error_gavel.png" alt="Workload3" style="width: 500px;">
    </div>
</div>
 -->
\begin{figure}[h]
    \centering
    \begin{minipage}{0.45\textwidth}
        \centering
        \includegraphics[width=\linewidth]{evaluated_pareto_front_avg_jct_avg_pred_error_themis1.png}
        \caption*{Workload A (light-tailed)}
    \end{minipage}\hfill
    \begin{minipage}{0.45\textwidth}
        \centering
        \includegraphics[width=\linewidth]{evaluated_pareto_front_avg_jct_avg_pred_error_gavel.png}
        \caption*{Workload B (heavy-tailed)}
    \end{minipage}
\end{figure}


#### Main trends/observations
* WFQs are able to achieve a higher quality Pareto-front compared to BOOST for both workloads
* The difference in quality is larger in the case of the heavy-tailed workload since the BOOST policy is mainly designed for light-tailed workloads
* For workload A, WFQ achieves 5% avg prediction error at a 1.12x performance cost while BOOST achieves the same at a 1.21x cost
* For workload B, WFQ achieves 5% avg prediction error at a 1.1x performance cost while BOOST achieves the same at a 2x cost